<a href="https://colab.research.google.com/github/Saransh-Basu-01/Advanced-Pytorch/blob/main/Implementing_the_training_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Iterating Through Data with DataLoader

In [ ]:
# Assume these are already defined and configured:
# train_dataloader = DataLoader(your_dataset, batch_size=64, shuffle=True)
# model = YourNeuralNetwork()
# loss_fn = torch.nn.CrossEntropyLoss() # Example loss function
# optimizer = torch.optim.SGD(model.parameters(), lr=0.01) # Example optimizer
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device) # Ensure model is on the correct device

num_epochs = 10 # Example number of passes over the dataset

# Outer loop for epochs
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}\n-------------------------------")

    # Set the model to training mode.
    # This enables features like dropout and batch normalization updates.
    model.train()

    # Inner loop for batches within an epoch
    # Iterate over batches provided by the DataLoader
    for batch_idx, data_batch in enumerate(train_dataloader):
        # 1. Unpack the batch
        # The structure depends on your Dataset's __getitem__ method.
        # For supervised learning, it's commonly (inputs, labels).
        inputs, labels = data_batch

        # 2. Move data to the target device (GPU or CPU)
        # This MUST match the device where the model resides.
        inputs = inputs.to(device)
        labels = labels.to(device)

        # ---> The next steps (forward pass, loss, backprop, optimize) <---
        # ---> using 'inputs' and 'labels' happen here.             <---
        # (These are detailed in the subsequent sections)

        # Example placeholder for where subsequent logic goes:
        # predictions = model(inputs)
        # loss = loss_fn(predictions, labels)
        # optimizer.zero_grad()
        # loss.backward()
        # optimizer.step()

        # Optional: Print progress periodically
        if batch_idx % 100 == 0:
            current_batch_size = len(inputs) # Get size of the current batch
            # Replace 0.0 with the actual calculated loss for logging
            current_loss = 0.0
            print(f"  Batch {batch_idx}: [{current_batch_size} samples] Current Loss: {current_loss:.4f}") # Example log

    # ---> Evaluation loop on validation data often follows here <---
    # (We'll cover evaluation loops later in this chapter)

print("Training finished!")


# The Forward Pass: Getting Predictions

In [ ]:
# Assume 'model' is an instance of your nn.Module subclass
# Assume 'data_batch' is loaded from the DataLoader
# Assume 'device' is defined (e.g., 'cuda' or 'cpu')

# Unpack the batch (adjust based on your DataLoader structure)
inputs, labels = data_batch
inputs = inputs.to(device) # Move input data to the correct device
labels = labels.to(device) # Move labels to the correct device (needed for loss)

# --- The Forward Pass ---
# Pass inputs through the model
outputs = model(inputs)
# -----------------------

# 'outputs' now contains the model's predictions for the 'inputs' batch.
# The next step will be to calculate the loss using these 'outputs' and 'labels'.


# Structure of Evaluation loop

In [ ]:
import torch

def evaluate_model(model, dataloader, criterion, device):
    """Evaluates the model on the provided dataset."""
    model.eval()  # Set model to evaluation mode
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():  # Disable gradient calculations
        for inputs, targets in dataloader:
            # Move data to the same device as the model
            inputs = inputs.to(device)
            targets = targets.to(device)

            # Forward pass
            outputs = model(inputs)

            # Calculate loss (optional, but useful for monitoring)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * inputs.size(0) # Accumulate batch loss

            # Calculate accuracy (example for classification)
            _, predicted_labels = torch.max(outputs, dim=1)
            correct_predictions += (predicted_labels == targets).sum().item()
            total_samples += targets.size(0)

    # Calculate average loss and accuracy for the entire dataset
    average_loss = total_loss / total_samples
    accuracy = correct_predictions / total_samples

    model.train() # Switch back to train mode if needed later
    return average_loss, accuracy

# --- Usage Example ---
# Assuming you have:
# model: Your nn.Module model
# validation_loader: Your DataLoader for the validation set
# criterion: Your loss function (e.g., nn.CrossEntropyLoss)
# device: torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# val_loss, val_accuracy = evaluate_model(model, validation_loader, criterion, device)
# print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')


# Saving and Loading Model Checkpoints

In [ ]:
# Assume model, optimizer, epoch, loss are defined
# PATH = "path/to/your/checkpoint.pth" # Define your save path

checkpoint = {
    'epoch': epoch + 1, # Save the next epoch number to start from
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss, # Or maybe validation loss
    # Add any other metrics or info you want to save
    # 'validation_accuracy': val_acc,
}

torch.save(checkpoint, PATH)
print(f"Checkpoint saved at epoch {epoch} to {PATH}")


In [ ]:
# First, instantiate your model structure
model = YourModelClass(*args, **kwargs)
# Define the path to your saved checkpoint
PATH = "path/to/your/checkpoint.pth"

# Load the checkpoint dictionary
checkpoint = torch.load(PATH)

# Load the model state dictionary from the checkpoint
model.load_state_dict(checkpoint['model_state_dict'])

# Set the model to evaluation mode
model.eval()

# Now the model is ready for inference
# with torch.no_grad():
#     outputs = model(inputs)


In [ ]:
# Instantiate model and optimizer first
model = YourModelClass(*args, **kwargs)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) # Or your chosen optimizer

# Define the path
PATH = "path/to/your/checkpoint.pth"
start_epoch = 0
best_loss = float('inf') # Example: initialize best loss

# Check if checkpoint exists to load from
if os.path.exists(PATH):
    checkpoint = torch.load(PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    best_loss = checkpoint['loss'] # Load previous loss
    print(f"Checkpoint loaded. Resuming training from epoch {start_epoch}")

# Set the model to training mode
model.train()

# Now you can continue your training loop, starting from start_epoch
# for epoch in range(start_epoch, num_epochs):
#     # ... training steps ...


In [ ]:
# Load a GPU-trained model onto a CPU
checkpoint = torch.load(PATH, map_location=torch.device('cpu'))
model.load_state_dict(checkpoint['model_state_dict'])

# Load any model onto the currently available device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
# Remember to also move your model to the device
model.to(device)
